# Housing Benefit

Housing Benefit is a means-tested DWP benefit that helps people on a low income with their rent. It is a legacy benefit: new working-age claimants have been routed to the Universal Credit housing element for years, and the remaining working-age awards outside specified and temporary accommodation were abolished on 1 July 2026 in Great Britain and 1 October 2026 in Northern Ireland. Housing Benefit continues for pensioners, protected mixed-age couples and tenants of specified (supported) or temporary accommodation, and PolicyEngine models it alongside the UC housing element so reform analyses can move between the two systems cleanly.

Parameters live in `policyengine_uk/parameters/gov/dwp/housing_benefit/` and the formulas in `policyengine_uk/variables/gov/dwp/housing_benefit/`.

## Eligibility

`housing_benefit_eligible` requires the benunit to be either in **social housing** or eligible for the **Local Housing Allowance** rate (broadly, private renters), with assessable capital at or below the working-age (or pension-age) capital limit, and to reach Housing Benefit by one of two routes:

- **Pension age**: every adult in the benunit has reached the qualifying age for State Pension Credit (State Pension age). New Housing Benefit claims are open to them ([SI 2014/1230 reg. 6A(4)](https://www.legislation.gov.uk/uksi/2014/1230/regulation/6A)), and Universal Credit is not, so neither a previous claim nor `would_claim_uc` matters.
- **Continuing award**: any other benunit was **already claiming** Housing Benefit (proxied by a positive `housing_benefit_reported`) and is **not claiming Universal Credit**. Working-age and mixed-age families cannot make new claims; they claim the Universal Credit housing element instead. For mixed-age couples the reported award stands in for the saving for couples who were entitled to Housing Benefit or Pension Credit on 14 May 2019 ([SI 2019/37 art. 4](https://www.legislation.gov.uk/uksi/2019/37/article/4)).

The pension-age test counts the same benunit members as Universal Credit's working-age test, and the continuing-award route requires not claiming Universal Credit, so no benunit receives both.

### Abolition of working-age awards

Working-age Housing Benefit awards outside specified or temporary accommodation were abolished from **1 July 2026** in Great Britain ([SI 2025/1148 art. 7](https://www.legislation.gov.uk/uksi/2025/1148/article/7), inserted by [SI 2026/409](https://www.legislation.gov.uk/uksi/2026/409/made)) and **1 October 2026** in Northern Ireland ([SR 2025/176 art. 7](https://www.legislation.gov.uk/nisr/2025/176/article/7), inserted by SR 2026/115). Claimants within [SI 2014/1230 reg. 6A(2)-(5)](https://www.legislation.gov.uk/uksi/2014/1230/regulation/6A) keep their awards: pension age, mixed-age couples protected since 14 May 2019, and specified or temporary accommodation. So do prisoners until their Universal Credit restriction ends and some income-related ESA claimants with an appointee.

The dates live in `gov.dwp.housing_benefit.working_age_awards_payable` (`great_britain` and `northern_ireland`), with 1 while working-age awards are payable and 0 after. PolicyEngine UK evaluates a year's parameters as one value for the fiscal year starting 6 April. By default that value is the one in force on 30 April, which would have left the whole of 2026-27 unaffected. This parameter is instead day-weighted across the fiscal year (`fiscal_year_blend`), so 2026 (2026-27) takes the share of the year before the abolition: 86/365 in Great Britain (6 April to 30 June) and 178/365 in Northern Ireland (6 April to 30 September). From 2027 it is 0.

`housing_benefit_payable_share` applies this per benunit. It is 1 for any benunit with a member over State Pension age, which stands in for the pension-age and protected mixed-age savings. In law a mixed-age couple is saved only if the older member claims under the pension-age rules and the couple has been entitled since 14 May 2019 ([SI 2019/37 art. 4](https://www.legislation.gov.uk/uksi/2019/37/article/4)), which the data cannot show. For other benunits it is the share for their country. A working-age continuing award stays eligible while the share is above 0, and `housing_benefit_pre_benefit_cap` pays the entitlement times the share. A working-age continuing award in 2026 is therefore paid for the part of the year before the abolition, and there is none from 2027.

What happens to the families whose awards end is set by `would_claim_uc`, the Universal Credit take-up input. Families that claim Universal Credit already receive it instead of Housing Benefit in every year. Families that do not claim it lose Housing Benefit and receive no Universal Credit. The Enhanced FRS take-up draw does not look at legacy receipt, so it leaves too many former claimants with neither. DWP's Move to Universal Credit statistics show that 97% of households sent a migration notice while on Housing Benefit and another legacy benefit claimed Universal Credit (98% of those also on Employment and Support Allowance), and 75% of Housing Benefit-only households did. Transitional protection, which went only to households that claimed after a migration notice, is not modelled.

The savings PolicyEngine cannot observe are not modelled: specified and temporary accommodation (open to new claims at any age, including alongside Universal Credit), prisoners, and the ESA appointee saving. The Enhanced FRS therefore has no working-age Housing Benefit from 2027. DWP's Spring 2026 forecast for Great Britain is £1.64bn of temporary-accommodation and £3.94bn of supported-accommodation Housing Benefit in 2026-27, across all ages.

**Guarantee Credit passport.** If the claimant or partner is in receipt of a guarantee credit (`in_receipt_of_guarantee_credit`: the benunit is eligible for Pension Credit, claims it, and its guarantee credit is positive, so the Pension Credit paid includes a guarantee credit), the whole of their capital and income is disregarded ([SI 2006/214 reg. 26](https://www.legislation.gov.uk/uksi/2006/214/regulation/26); Northern Ireland: [SR 2006/406 reg. 24](https://www.legislation.gov.uk/nisr/2006/406/regulation/24)). `housing_benefit_assessable_capital`, `housing_benefit_tariff_income` and `housing_benefit_applicable_income` are then zero, so the capital limit cannot bite and no income is tapered. A guarantee credit computed for a family that does not claim Pension Credit (`would_claim_pc`), or that PolicyEngine does not treat as eligible for it (`is_pension_credit_eligible`), does not count. `is_pension_credit_eligible` omits the saving in [SI 2019/37 art. 4](https://www.legislation.gov.uk/uksi/2019/37/article/4) for a mixed-age couple entitled to Pension Credit or Housing Benefit as that couple on 14 May 2019 (and to either ever since). Such a couple may be entitled to a guarantee credit in law and, if they receive it, are passported; PolicyEngine does not passport them until that saving is modelled.

Take-up is handled by `would_claim_housing_benefit`, which is true for benunits that report Housing Benefit and, when no reported benefits are supplied (as in household calculations), for everyone. In the Enhanced FRS take-up therefore follows reported claims.

## How PolicyEngine computes Housing Benefit

The award is computed in four stages:

1. **Eligible rent** (`benunit_rent`): the contracted rent for the property, capped at the Local Housing Allowance rate where the LHA applies.
2. **Applicable amount** (`housing_benefit_applicable_amount`): a personal allowance derived from the benunit's composition (single, couple, lone parent) and age, plus any disability/carer premiums. This is the "minimum income" the household needs before HB tapers begin.
3. **Means test** (`housing_benefit_entitlement`): the benunit's `housing_benefit_applicable_income` in excess of the applicable amount is tapered against the eligible rent at the published `withdrawal_rate` (currently 0.65). The result is then reduced by `housing_benefit_non_dep_deductions` for any non-dependant adults living in the household.
4. **Benefit cap** (`housing_benefit`): the post-means-test award is finally reduced by `benefit_cap_reduction` if the household's total benefit income exceeds the cap and no cap exemption applies.

The take-up gate (`would_claim_housing_benefit`) is applied inside `housing_benefit_pre_benefit_cap`.

## Personal allowances

The applicable amount uses age- and household-composition bands. The chart below shows how the weekly allowances have evolved.

Since 1 April 2021 the pension-age allowances depend on when members of the benunit attained State Pension age ([SI 2006/214 Sch. 3 para. 1](https://www.legislation.gov.uk/uksi/2006/214/schedule/3), as amended by [SI 2021/188](https://www.legislation.gov.uk/uksi/2021/188/regulation/2/made); Northern Ireland: [SR 2006/406 Sch. 4 para. 1](https://www.legislation.gov.uk/nisr/2006/406/schedule/4)). A single claimant or lone parent who attained it before 1 April 2021, or a couple in which either member did, keeps the higher allowance (`aged`, £256.00 single and £383.35 couple a week in 2026-27). Where every member attained it on or after that date, the lower allowance applies (`aged_from_cutoff`, £238.00 and £363.25). `housing_benefit_attained_pension_age_before_cutoff` compares each member's `state_pension_age_date`, the day set by their date of birth under the Pensions Act 1995 timetable, with `pension_age_cutoff`. The regulations do not settle a couple in which one member has not yet reached State Pension age and the other reached it after the cutoff; PolicyEngine gives it the lower allowance, following the explanatory note to SI 2021/188.

Council Tax Reduction reuses these allowances. England's pensioner scheme has the same split and amounts ([SI 2012/2885 Sch. 2 para. 1](https://www.legislation.gov.uk/uksi/2012/2885/schedule/2), as amended by [SI 2021/29](https://www.legislation.gov.uk/uksi/2021/29/contents/made)). The Scottish ([SSI 2012/319 Sch. 1 para. 2](https://www.legislation.gov.uk/ssi/2012/319/schedule/1)) and Welsh ([SI 2013/3029 Sch. 2 para. 1](https://www.legislation.gov.uk/wsi/2013/3029/schedule/2)) schemes keep one pension-age rate, the higher one.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from policyengine_uk.system import system
import pandas as pd
import plotly.express as px
from policyengine_core.charts import format_fig

allowances = system.parameters.gov.dwp.housing_benefit.allowances

params = {
    "Single (younger)": allowances.single.younger,
    "Single (older)": allowances.single.older,
    "Single (pension age)": allowances.single.aged,
    "Single (pension age, reached from April 2021)": allowances.single.aged_from_cutoff,
    "Lone parent (younger)": allowances.lone_parent.younger,
    "Lone parent (older)": allowances.lone_parent.older,
    "Lone parent (pension age)": allowances.lone_parent.aged,
    "Couple (younger)": allowances.couple.younger,
    "Couple (older)": allowances.couple.older,
    "Couple (pension age)": allowances.couple.aged,
    "Couple (pension age, both reached from April 2021)": allowances.couple.aged_from_cutoff,
}

dates = [f"{y}-04-01" for y in range(2016, 2027)]
rows = [
    {"date": d, "composition": name, "weekly allowance": p(d)}
    for d in dates
    for name, p in params.items()
]
df = pd.DataFrame(rows)

fig = (
    px.line(
        df,
        x="date",
        y="weekly allowance",
        color="composition",
        title="Housing Benefit weekly personal allowances",
    )
    .update_layout(
        yaxis_tickprefix="£",
        yaxis_tickformat=",.2f",
        xaxis_title="Date",
        legend_title="Composition",
    )
    .update_traces(line_shape="hv")
)
fig = format_fig(fig)
fig

## References

- [Social Security Contributions and Benefits Act 1992, s. 130](https://www.legislation.gov.uk/ukpga/1992/4/section/130) — the primary statute creating Housing Benefit.
- [The Housing Benefit Regulations 2006 (SI 2006/213)](https://www.legislation.gov.uk/uksi/2006/213/contents) and [The Housing Benefit (Persons who have attained the qualifying age for state pension credit) Regulations 2006 (SI 2006/214)](https://www.legislation.gov.uk/uksi/2006/214/contents) — the operational regulations for working-age and pension-age claimants respectively.
- DWP, [Housing Benefit: detailed information](https://www.gov.uk/housing-benefit) — user-facing guidance covering the move to Universal Credit.
- DWP, [Housing Benefit caseload statistics](https://www.gov.uk/government/collections/housing-benefit-caseload-statistics) — official caseload and expenditure outturns used by `policyengine-uk-data` for calibration.